In [ ]:
"""
================================================================================
MONKEYPOX SEVERITY CLASSIFICATION
Multi-Segmentation Comparison: fixed_global | otsu | adaptive | none
Kaggle version — dataset: dirework/mpox-datasett
================================================================================
"""

import os, sys, glob, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from tqdm import tqdm
warnings.filterwarnings('ignore')

np.random.seed(42); tf.random.set_seed(42)

print("=" * 80)
print("MONKEYPOX SEVERITY — MULTI-SEGMENTATION COMPARISON")
print(f"TensorFlow Version: {tf.__version__}")
print("=" * 80)

# ============================================================================
# 0. AUTO-LOCATE DATASET
# ============================================================================
def find_data_dir(root='/kaggle/input'):
    """Search for a folder whose children include '1_*'."""
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.startswith('1_'):
                return dirpath
    return None

DATASET_PATH = find_data_dir()
if DATASET_PATH is None:
    for c in ['/kaggle/input/mpox-datasett/dataset',
              '/kaggle/input/mpox-datasett/mpoxdataset',
              '/kaggle/input/mpox-datasett',
              '/kaggle/input/mpox-dataset/dataset',
              '/kaggle/input/mpox-dataset/mpoxdataset',
              '/kaggle/input/mpox-dataset']:
        if os.path.isdir(c):
            DATASET_PATH = c; break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Run in a separate cell:\n"
        "import os\n"
        "for dp, dn, fn in os.walk('/kaggle/input'): print(dp, '->', dn[:5])"
    )

print(f"\n[1] Dataset located at: {DATASET_PATH}")
print("    Contents:", sorted(os.listdir(DATASET_PATH))[:10])

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================================
# 2. CLASS LABELS
# ============================================================================
class_names = ['Macules', 'Papules', 'Vesicles', 'Pustules', 'Scubs', 'Normal']
class_mapping = {
    '1_Macules': 0, '2_Papules': 1, '3_Vesicles': 2,
    '4_Pustules': 3, '5_Scubs': 4, '6_Normal': 5,
}

# Fallback: accept 5_Scabs spelling too
if not os.path.isdir(os.path.join(DATASET_PATH, '5_Scubs')):
    if os.path.isdir(os.path.join(DATASET_PATH, '5_Scabs')):
        class_mapping = {
            '1_Macules': 0, '2_Papules': 1, '3_Vesicles': 2,
            '4_Pustules': 3, '5_Scabs': 4, '6_Normal': 5,
        }

# ============================================================================
# 3. LOAD DATASET (ONCE)
# ============================================================================
print("\n[2] Loading Dataset...")
images, labels, image_paths = [], [], []

for folder_name, label in class_mapping.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    if not os.path.exists(folder_path):
        print(f"  [WARN] Missing: {folder_path}"); continue
    files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        files.extend(glob.glob(os.path.join(folder_path, ext)))
    print(f"  {folder_name}: {len(files)} images")
    for p in tqdm(files, desc=f"  Loading {folder_name}"):
        img = cv2.imread(p)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        images.append(img); labels.append(label); image_paths.append(p)

images = np.array(images, dtype=np.float32) / 255.0
labels = np.array(labels)

print(f"\nTotal: {len(images)} images, shape {images[0].shape}")
for i, c in enumerate(class_names):
    n = int(np.sum(labels == i))
    print(f"  {c}: {n} ({n/len(labels)*100:.1f}%)")

# ============================================================================
# 4. TRAIN / VAL / TEST SPLIT (SHARED ACROSS ALL METHODS)
# ============================================================================
print("\n[3] Splitting 70/15/15 (shared across all segmentation methods)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"  Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

# ============================================================================
# 5. SEGMENTATION — 4 METHODS
# ============================================================================
class ThresholdSegmentation:
    def __init__(self, method='adaptive', fixed_T=125):
        self.method = method
        self.fixed_T = fixed_T

    def segment_image(self, image):
        if self.method == 'none':
            return image

        img_u8 = (image * 255).astype(np.uint8) if image.dtype != np.uint8 else image
        gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY) if img_u8.ndim == 3 else img_u8
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        if self.method == 'fixed_global':
            _, mask = cv2.threshold(blurred, self.fixed_T, 255, cv2.THRESH_BINARY)
        elif self.method == 'otsu':
            _, mask = cv2.threshold(blurred, 0, 255,
                                    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif self.method == 'adaptive':
            mask = cv2.adaptiveThreshold(blurred, 255,
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, 11, 2)
        else:
            raise ValueError(f"Unknown method: {self.method}")

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)

        segmented = cv2.bitwise_and(img_u8, img_u8, mask=mask)
        return segmented.astype(np.float32) / 255.0


SEG_METHODS = ['fixed_global', 'otsu', 'adaptive', 'none']

def apply_segmentation(X, method):
    seg = ThresholdSegmentation(method=method)
    return np.array([seg.segment_image(img) for img in tqdm(X, desc=f"  Seg[{method}]")])

# ============================================================================
# 6. CNN MODEL
# ============================================================================
def build_model(input_shape=(224, 224, 3), num_classes=6, lr=0.001):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x); x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x); x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x); x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x); x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(512, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(512, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ============================================================================
# 7. TRAIN ONE MODEL PER SEGMENTATION METHOD
# ============================================================================
EPOCHS     = 100    
BATCH_SIZE = 32

results = {}

for method in SEG_METHODS:
    print("\n" + "=" * 80)
    print(f"TRAINING WITH SEGMENTATION: {method.upper()}")
    print("=" * 80)

    Xtr = apply_segmentation(X_train, method)
    Xva = apply_segmentation(X_val,   method)
    Xte = apply_segmentation(X_test,  method)

    model = build_model()

    ckpt = os.path.join(OUT_DIR, f'best_{method}.h5')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=10, min_lr=1e-7, verbose=1),
        ModelCheckpoint(ckpt, monitor='val_accuracy',
                        save_best_only=True, verbose=1),
    ]

    history = model.fit(
        Xtr, y_train,
        validation_data=(Xva, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1)

    test_loss, test_acc = model.evaluate(Xte, y_test, verbose=0)
    y_proba = model.predict(Xte, verbose=0)
    y_pred  = np.argmax(y_proba, axis=1)

    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)

    results[method] = {
        'accuracy':  test_acc,
        'precision': prec,
        'recall':    rec,
        'f1':        f1,
        'report':    classification_report(y_test, y_pred,
                                           target_names=class_names,
                                           output_dict=True, zero_division=0),
        'history':   history.history,
        'y_true':    y_test,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'model':     model,
    }

    model.save(os.path.join(OUT_DIR, f'mpoxsevnet_{method}.h5'))

    print(f"\n[{method}] Test Acc: {test_acc*100:.2f}% | "
          f"Prec: {prec*100:.2f}% | Rec: {rec*100:.2f}% | F1: {f1*100:.2f}%")

# ============================================================================
# 8. COMPARISON TABLE
# ============================================================================
print("\n" + "=" * 80)
print("SEGMENTATION COMPARISON TABLE")
print("=" * 80)

comparison = pd.DataFrame({
    m: {
        'Accuracy':  f"{results[m]['accuracy']*100:.2f}%",
        'Precision': f"{results[m]['precision']*100:.2f}%",
        'Recall':    f"{results[m]['recall']*100:.2f}%",
        'F1-score':  f"{results[m]['f1']*100:.2f}%",
    } for m in SEG_METHODS
}).T.loc[SEG_METHODS]

print(comparison.to_string())
comparison.to_csv(os.path.join(OUT_DIR, 'segmentation_comparison.csv'))

numeric = pd.DataFrame({
    m: {
        'Accuracy':  results[m]['accuracy'],
        'Precision': results[m]['precision'],
        'Recall':    results[m]['recall'],
        'F1-score':  results[m]['f1'],
    } for m in SEG_METHODS
}).T.loc[SEG_METHODS]
numeric.to_csv(os.path.join(OUT_DIR, 'segmentation_comparison_numeric.csv'))

# ============================================================================
# 9. BAR CHART  (FIXED: explicit label↔key mapping, no string surgery)
# ============================================================================
print("\n[9] Plotting comparison bar chart...")

metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-score']
metric_keys   = ['accuracy',  'precision', 'recall', 'f1']   # matches results dict

x = np.arange(len(SEG_METHODS))
width = 0.20
colors = ['#2ecc71', '#3498db', '#f39c12', '#9b59b6']

plt.figure(figsize=(12, 6))
for i, (label, key) in enumerate(zip(metric_labels, metric_keys)):
    vals = [results[m][key] * 100 for m in SEG_METHODS]
    bars = plt.bar(x + (i - 1.5) * width, vals, width,
                   label=label, color=colors[i],
                   edgecolor='black', linewidth=1.2, alpha=0.85)
    for b, v in zip(bars, vals):
        plt.text(b.get_x() + b.get_width()/2, v + 0.8,
                 f"{v:.1f}", ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

plt.xticks(x, SEG_METHODS, fontsize=12, fontweight='bold')
plt.ylabel('Percentage (%)', fontsize=13, fontweight='bold')
plt.title('Segmentation Method Comparison — MPoxSevNet',
          fontsize=15, fontweight='bold')
plt.ylim(0, 105)
plt.grid(True, alpha=0.3, axis='y')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'segmentation_comparison.png'), dpi=150)
plt.show()

# ============================================================================
# 10. PER-METHOD TRAINING CURVES
# ============================================================================
print("\n[10] Plotting per-method training curves...")

fig, axes = plt.subplots(2, len(SEG_METHODS),
                         figsize=(5 * len(SEG_METHODS), 10))
for j, m in enumerate(SEG_METHODS):
    h = results[m]['history']
    axes[0, j].plot(h['accuracy'],     label='Train', color='blue', lw=2)
    axes[0, j].plot(h['val_accuracy'], label='Val',   color='red',  lw=2)
    axes[0, j].set_title(f'{m} — Accuracy', fontweight='bold')
    axes[0, j].set_xlabel('Epoch'); axes[0, j].set_ylabel('Accuracy')
    axes[0, j].legend(); axes[0, j].grid(alpha=0.3); axes[0, j].set_ylim(0, 1)

    axes[1, j].plot(h['loss'],     label='Train', color='blue', lw=2)
    axes[1, j].plot(h['val_loss'], label='Val',   color='red',  lw=2)
    axes[1, j].set_title(f'{m} — Loss', fontweight='bold')
    axes[1, j].set_xlabel('Epoch'); axes[1, j].set_ylabel('Loss')
    axes[1, j].legend(); axes[1, j].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves_per_method.png'), dpi=150)
plt.show()

# ============================================================================
# 11. PER-METHOD CONFUSION MATRICES
# ============================================================================
print("\n[11] Plotting per-method confusion matrices...")

fig, axes = plt.subplots(1, len(SEG_METHODS),
                         figsize=(5.5 * len(SEG_METHODS), 5))
for j, m in enumerate(SEG_METHODS):
    cm = confusion_matrix(results[m]['y_true'], results[m]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[j], cbar=False, annot_kws={'size': 9})
    axes[j].set_title(f'{m}\nAcc = {results[m]["accuracy"]*100:.2f}%',
                      fontweight='bold')
    axes[j].set_xlabel('Predicted'); axes[j].set_ylabel('True')
    axes[j].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrices_per_method.png'), dpi=150)
plt.show()

# ============================================================================
# 12. PER-METHOD ROC CURVES
# ============================================================================
print("\n[12] Plotting per-method ROC curves...")

fig, axes = plt.subplots(1, len(SEG_METHODS),
                         figsize=(5.5 * len(SEG_METHODS), 5))
y_test_oh = to_categorical(results[SEG_METHODS[0]]['y_true'], num_classes=6)
roc_colors = ['#1f77b4', '#ff7f0e', '#2ca02c',
              '#d62728', '#9467bd', '#8c564b']

for j, m in enumerate(SEG_METHODS):
    y_proba = results[m]['y_proba']
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_test_oh[:, i], y_proba[:, i])
        axes[j].plot(fpr, tpr, lw=2, color=roc_colors[i],
                     label=f'{cname} ({auc(fpr, tpr):.2f})')
    axes[j].plot([0, 1], [0, 1], 'k--', lw=1.5)
    axes[j].set_title(f'{m} — ROC', fontweight='bold')
    axes[j].set_xlabel('FPR'); axes[j].set_ylabel('TPR')
    axes[j].legend(fontsize=8, loc='lower right')
    axes[j].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'roc_curves_per_method.png'), dpi=150)
plt.show()

# ============================================================================
# 13. SAVE SUMMARY JSON
# ============================================================================
summary = {m: {k: float(results[m][k]) for k in
               ['accuracy', 'precision', 'recall', 'f1']}
           for m in SEG_METHODS}
with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("✅ ALL DONE")
print(f"Outputs saved to: {OUT_DIR}")
print("=" * 80)
print("\nFinal comparison table:")
print(comparison.to_string())
print("\nFiles in /kaggle/working:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}")